# 🚔 Cincinnati Public Safety Incident Analysis

This notebook analyzes reported public safety incidents in Cincinnati using Python. The project focuses on cleaning incident data, mapping geographic patterns, and building a simple baseline model to classify incident categories.

## Project Goals
- Clean and prepare incident-level public safety data
- Visualize where incidents occur across Cincinnati
- Explore geographic and time-based features
- Build a baseline KNN classification model
- Summarize insights for public safety and city operations use cases


## 1. Import Libraries

The required packages are listed in `requirements.txt`. To install them locally, run:

```bash
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import plugins

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Load Data

This notebook uses a relative file path so it can run from the GitHub repository without relying on a local computer path.

Expected structure:

```text
project-folder/
├── incident-trend-analysis.ipynb
├── data/
│   └── Reported_Crime__STARS_Category_Offenses__20250524.csv
└── outputs/
```


In [ ]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Update the file name below if your CSV has a different name.
data_file = DATA_DIR / "Reported_Crime__STARS_Category_Offenses__20250524.csv"

if not data_file.exists():
    available_csvs = list(DATA_DIR.glob("*.csv")) if DATA_DIR.exists() else []
    if available_csvs:
        data_file = available_csvs[0]
        print(f"Using detected CSV file: {data_file}")
    else:
        raise FileNotFoundError(
            "No CSV file found. Place the incident dataset inside the data/ folder."
        )

df = pd.read_csv(data_file)
print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns.")
df.head()


## 3. Data Cleaning

For mapping and modeling, rows without valid latitude or longitude values are removed. A sample is used to keep the interactive map lightweight for GitHub and browser viewing.


In [ ]:
coordinate_cols = ["LATITUDE_X", "LONGITUDE_X"]
required_cols = coordinate_cols + ["STARS_CATEGORY"]

missing_required = [col for col in required_cols if col not in df.columns]
if missing_required:
    raise KeyError(f"Missing required columns: {missing_required}")

sample_size = min(10_000, len(df))
df_sample = df.sample(sample_size, random_state=42)

df_clean = df_sample.dropna(subset=coordinate_cols).copy()
df_clean = df_clean[df_clean["STARS_CATEGORY"].notna()].copy()

print(f"Rows before sampling: {len(df):,}")
print(f"Rows sampled: {len(df_sample):,}")
print(f"Rows after cleaning: {len(df_clean):,}")


## 4. Interactive Incident Map

The map below clusters incidents by location, making it easier to identify high-activity areas across Cincinnati.


In [ ]:
cincinnati_map = folium.Map(location=[39.1031, -84.5120], zoom_start=12)
incident_cluster = plugins.MarkerCluster().add_to(cincinnati_map)

for lat, lng, category in zip(
    df_clean["LATITUDE_X"],
    df_clean["LONGITUDE_X"],
    df_clean["STARS_CATEGORY"]
):
    folium.Marker(
        location=[lat, lng],
        popup=str(category)
    ).add_to(incident_cluster)

map_path = OUTPUT_DIR / "cincinnati_incident_map.html"
cincinnati_map.save(map_path)
print(f"Interactive map saved to: {map_path}")

cincinnati_map


## 5. Feature Selection

The model uses location and time-based variables to predict the incident category. This is a simple baseline model, not a production-ready public safety model.


In [ ]:
target_col = "STARS_CATEGORY"
feature_cols = ["LATITUDE_X", "LONGITUDE_X", "Hour_From", "Hour_To"]

available_features = [col for col in feature_cols if col in df_clean.columns]
missing_features = sorted(set(feature_cols) - set(available_features))

if missing_features:
    print(f"Warning: Missing feature columns excluded from model: {missing_features}")

X = df_clean[available_features].fillna(0)
y = df_clean[target_col]

print(f"Using features: {available_features}")
print(f"Target classes: {y.nunique()}")


## 6. Train-Test Split and Scaling

The dataset is split into training and test sets. Numeric features are standardized before training the KNN model.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 7. Baseline KNN Model

A K-Nearest Neighbors classifier is used as a baseline to test whether location and time features contain predictive signal for incident category classification.


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))


## 8. Confusion Matrix

The confusion matrix helps evaluate which incident categories the model classifies well and where misclassification occurs.


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=False,
    xticklabels=labels,
    yticklabels=labels
)
plt.ylabel("Actual Category")
plt.xlabel("Predicted Category")
plt.title("KNN Confusion Matrix")
plt.tight_layout()
plt.show()


## 9. Conclusion

This project demonstrates how public safety incident data can be cleaned, visualized, and modeled to better understand local activity patterns. The interactive map provides a geographic overview of incident concentration, while the baseline model explores whether basic location and time variables can predict incident categories.

## Potential Next Steps
- Add neighborhood or district-level aggregation
- Compare incident trends by month, weekday, and hour
- Create additional charts for top incident categories
- Test additional models such as Random Forest or XGBoost
- Build a dashboard view for non-technical stakeholders
